# 02 - Exploratory Data Analysis - Germany Used Cars 2023

**Project question:** do electric cars lose value faster than combustion cars, at the same age and mileage?

In notebook 01 we cleaned the data. Here we **get to know it** before testing anything:
1. Describe each variable alone (univariate).
2. Relate price to age, mileage and fuel (bivariate).
3. Look for **confounders**: things that differ between electric and combustion cars and could fake (or hide) a difference in depreciation.
4. Write the insights in plain words. They decide how we build the test in notebook 03.

| Part | Content |
|---|---|
| 1 | Load the clean data |
| 2 | Univariate: brand, fuel, price, age, mileage |
| 3 | Price vs age: the key chart |
| 4 | Price vs mileage and the correlation matrix |
| 5 | Confounders: starting price, kilometres per year |
| 6 | Insights |

## Setup

In [1]:
import sys
sys.path.append("..")            # so we can import from src/

import numpy as np
import pandas as pd
import plotly.express as px

from src.cleaning import load_raw, clean
from src.metrics import median_price_by_age

pd.set_option("display.max_columns", None)
FUEL_ORDER = ["Petrol", "Diesel", "Hybrid", "Electric", "Other"]
FUEL_COLORS = {"Petrol": "#5B8DEF", "Diesel": "#9AA5B1", "Hybrid": "#FFB547", "Electric": "#00B386", "Other": "#B48CFF"}

In [2]:
df = clean(load_raw())          # same cleaning as notebook 01, from src/cleaning.py
print(df.shape)
df.head()

(244255, 17)


,brand,model,color,registration_date,year,price_in_euro,power_kw,power_ps,transmission_type,fuel_type,fuel_consumption_l_100km,fuel_consumption_g_km,mileage_in_km,offer_description,age_years,fuel_group,km_per_year
0,alfa-romeo,Alfa Romeo GTV,red,1995-10-01,1995,1300,148.0,201.0,Manual,Petrol,"10,9 l/100 km",260 g/km,160500.0,2.0 V6 TB,28.08,Petrol,5715.811966
1,alfa-romeo,Alfa Romeo 164,black,1995-02-01,1995,24900,191.0,260.0,Manual,Petrol,NaN,- (g/km),190000.0,"Q4 Allrad, 3.2L GTA",28.75,Petrol,6608.695652
2,alfa-romeo,Alfa Romeo Spider,black,1995-02-01,1995,5900,110.0,150.0,Unknown,Petrol,NaN,- (g/km),129000.0,ALFA ROME 916,28.75,Petrol,4486.956522
3,alfa-romeo,Alfa Romeo Spider,black,1995-07-01,1995,4900,110.0,150.0,Manual,Petrol,"9,5 l/100 km",225 g/km,189500.0,2.0 16V Twin Spark L,28.34,Petrol,6686.661962
4,alfa-romeo,Alfa Romeo 164,red,1996-11-01,1996,17950,132.0,179.0,Manual,Petrol,"7,2 l/100 km",- (g/km),96127.0,"3.0i Super V6, absoluter Topzustand !",27.00,Petrol,3560.259259


---
# 1. A first look at the clean data

In [3]:
df[["price_in_euro", "age_years", "mileage_in_km", "power_kw", "km_per_year"]].describe().round(0)

,price_in_euro,age_years,mileage_in_km,power_kw,km_per_year
count,244255.0,244255.0,244255.0,244130.0,244255.0
mean,26092.0,7.0,86395.0,127.0,11358.0
std,37348.0,5.0,75125.0,72.0,7547.0
min,500.0,0.0,0.0,1.0,0.0
25%,11900.0,3.0,26589.0,84.0,6526.0
50%,19690.0,6.0,69050.0,110.0,10475.0
75%,29900.0,10.0,128000.0,140.0,15077.0
max,5890500.0,29.0,500000.0,735.0,381356.0


Reading the table: the **median** price is about 19,700 EUR but the **mean** is 26,100 EUR, and the maximum is 5.9 million.
The distribution is **skewed to the right** (a few very expensive cars pull the mean up). For this reason we always use the **median**
and the **logarithm** of the price, never the raw mean.

---
# 2. Univariate analysis
One variable at a time: *which values, and how often?*

## 2.1 Brands

In [4]:
top_brands = df["brand"].value_counts().head(15).rename("listings").reset_index()
fig = px.bar(top_brands, x="listings", y="brand", orientation="h", title="Listings per brand (top 15)")
fig.update_layout(yaxis={"categoryorder": "total ascending"})
fig.show()
print("Brands in the file:", df["brand"].nunique())

Brands in the file: 47


Volkswagen, Mercedes-Benz, Audi, Opel and BMW lead the market. There are 47 brands, but many have very few cars: for brand-level
metrics we will always require a **minimum number of listings**.

## 2.2 Fuel types

In [5]:
fuel = df["fuel_group"].value_counts().reindex(FUEL_ORDER).rename("listings").to_frame()
fuel["share_%"] = (fuel["listings"] / fuel["listings"].sum() * 100).round(1)
fuel

,listings,share_%
fuel_group,,
Petrol,139421,57.1
Diesel,84738,34.7
Hybrid,12214,5.0
Electric,5870,2.4
Other,2012,0.8


Petrol (57%) and diesel (35%) are the market. Electric cars are only **2.4%** of the listings (5,870): enough to estimate a trend,
but we must be careful when we split them further (by brand, by age).

## 2.3 Electric cars are a recent story

In [6]:
df["reg_year"] = df["registration_date"].dt.year
ev_share = (df[df["reg_year"] >= 2010].groupby("reg_year")["fuel_group"]
              .apply(lambda s: s.eq("Electric").mean() * 100).rename("ev_share_%").reset_index())
fig = px.line(ev_share, x="reg_year", y="ev_share_%", markers=True,
              title="Share of electric cars by registration year (%)", labels={"reg_year": "registration year"})
fig.show()

Among cars registered in 2010-2016 electric cars are almost absent (0.5% or less); the share rises to 4.8% for 2020 and 8.5% for 2021, then falls to 7.3% (2022) and 5.7% (2023). This is why the electric
cars in our file are **young**, and also why the oldest ones are a different kind of car (early, short-range models).

## 2.4 Price, age and mileage

In [7]:
def binned_hist(series, bins, title, xlabel, log_y=False):
    # bin first: lighter than sending 244k rows to the browser
    counts, edges = np.histogram(series.dropna(), bins=bins)
    table = pd.DataFrame({xlabel: edges[:-1], "listings": counts})
    return px.bar(table, x=xlabel, y="listings", log_y=log_y, title=title)

binned_hist(df["price_in_euro"].clip(upper=150_000), 75, "Price (capped at 150,000 EUR for readability)", "price_in_euro").show()
binned_hist(df["age_years"], np.arange(0, 31, 1), "Age in years", "age_years").show()
binned_hist(df["mileage_in_km"], 50, "Mileage (km)", "mileage_in_km").show()

- **Price:** strongly right-skewed. Most cars are under 40,000 EUR.
- **Age:** many young cars (0-3 years) and a long tail up to 29 years.
- **Mileage:** many low-mileage cars (new and nearly new) and a long tail.

---
# 3. Price vs age: the key chart
For each year of age and each fuel group, the **median price** (only buckets with at least 30 cars).

In [8]:
curve = median_price_by_age(df, bin_width=1, max_age=15)
curve["fuel_group"] = pd.Categorical(curve["fuel_group"], categories=FUEL_ORDER, ordered=True)
fig = px.line(curve.sort_values(["fuel_group", "age_bucket"]), x="age_bucket", y="median_price", color="fuel_group",
              markers=True, color_discrete_map=FUEL_COLORS, log_y=True,
              title="Median price by age and fuel (log scale)",
              labels={"age_bucket": "age (years)", "median_price": "median price (EUR)", "fuel_group": ""})
fig.show()

In [9]:
table = curve[(curve["age_bucket"] <= 6.5)].pivot(index="age_bucket", columns="fuel_group", values="median_price").round(0)
table[["Petrol", "Diesel", "Hybrid", "Electric"]]

fuel_group,Petrol,Diesel,Hybrid,Electric
age_bucket,,,,
0.5,29990.0,50880.0,34955.0,46250.0
1.5,27890.0,42885.0,32664.0,41970.0
2.5,24554.0,33895.0,30490.0,28950.0
3.5,22999.0,29950.0,31111.0,25390.0
4.5,20990.0,26900.0,27970.0,19950.0
5.5,19190.0,23850.0,22990.0,17950.0
6.5,17500.0,20999.0,21500.0,15999.0


**What we see (log scale, so a straight line = a constant % loss per year):**
- Electric cars start **higher** than petrol (about 46,000 vs 30,000 EUR when under 1 year old) and their line is **steeper**.
- Between age 3 and 5 the order flips: at 4-5 years the electric median (about 20,000 EUR) is already *below* petrol (about 21,000 EUR),
  and at 5-6 years it is about 18,000 EUR against 19,000 (petrol) and 24,000 (diesel).
- The electric line is based on few cars after 6-7 years (it ends at 9-10 years, where the buckets are thin), so we do not trust it beyond that.

This is the picture of our hypothesis, but it is **not yet a proof**: electric cars also differ in mileage and brand mix. Next, the other variables.

---
# 4. Price vs mileage and correlations

In [10]:
sample = df[df["age_years"] <= 6].sample(8000, random_state=42)
fig = px.scatter(sample, x="mileage_in_km", y="price_in_euro", color="fuel_group", log_y=True, opacity=0.5,
                 color_discrete_map=FUEL_COLORS, category_orders={"fuel_group": FUEL_ORDER},
                 title="Price vs mileage, cars up to 6 years (random sample of 8,000)",
                 labels={"mileage_in_km": "mileage (km)", "price_in_euro": "price (EUR)", "fuel_group": ""})
fig.show()

In [11]:
corr_data = df[["price_in_euro", "age_years", "mileage_in_km", "power_kw"]].copy()
corr_data["price_in_euro"] = np.log(corr_data["price_in_euro"])
corr_data = corr_data.rename(columns={"price_in_euro": "log_price"})
corr = corr_data.corr(method="spearman").round(2)       # Spearman: no need for a straight-line relation
fig = px.imshow(corr, text_auto=True, color_continuous_scale="RdBu", zmin=-1, zmax=1,
                title="Spearman correlation")
fig.show()

- Price falls with **age** (-0.70) and **mileage** (-0.62), and rises with **power** (0.61).
- **Age and mileage are strongly linked** (0.83): older cars have more km. We cannot study age without controlling mileage, and the other way round.
  That is exactly the "same age and mileage" of our hypothesis.

---
# 5. Confounders: what else differs between electric and combustion cars?
## 5.1 Starting price (cars under 1 year old)

In [12]:
new_cars = df[(df["age_years"] <= 1) & df["fuel_group"].isin(["Petrol", "Diesel", "Hybrid", "Electric"])]
fig = px.box(new_cars, x="fuel_group", y="price_in_euro", color="fuel_group", points=False, log_y=True,
             category_orders={"fuel_group": ["Petrol", "Diesel", "Hybrid", "Electric"]},
             color_discrete_map=FUEL_COLORS, title="Price of cars up to 1 year old")
fig.update_layout(showlegend=False)
fig.show()
new_cars.groupby("fuel_group")["price_in_euro"].median()

fuel_group
Diesel      50250.0
Electric    45825.0
Hybrid      34990.0
Petrol      29990.0
Name: price_in_euro, dtype: float64

Almost-new electric cars cost much more than petrol ones (median 45,800 vs 30,000 EUR). If we measured depreciation in **euros**, electric cars would
lose more just because they start higher. For this reason the hypothesis uses **percentages** (the slope of log price): a loss of 10% per year
means the same for a 15,000 EUR car and a 50,000 EUR car.

## 5.2 Kilometres per year

In [13]:
usage = df[(df["age_years"].between(1, 6)) & df["fuel_group"].isin(["Petrol", "Diesel", "Hybrid", "Electric"])]
usage.groupby("fuel_group")["km_per_year"].median().round(0).rename("median_km_per_year")

fuel_group
Diesel      15249.0
Electric     6569.0
Hybrid       9210.0
Petrol       8889.0
Name: median_km_per_year, dtype: float64

In [14]:
print("Cars with more than 60,000 km per year:", (df["km_per_year"] > 60_000).sum(), f"({(df['km_per_year'] > 60_000).mean():.2%})")

Cars with more than 60,000 km per year: 153 (0.06%)


Electric cars are driven **much less** (about 6,600 km per year vs 8,900 for petrol and 15,200 for diesel). So at the same age they have fewer km,
and a naive comparison would make them look *better* than they are. This is the second reason to control for mileage.
(Only 153 cars (0.06%) have more than 60,000 km per year; they are kept because nothing proves them wrong, and with so few they cannot move the result.)

## 5.3 Transmission

In [15]:
pd.crosstab(df["fuel_group"], df["transmission_type"], normalize="index").mul(100).round(1)

transmission_type,Automatic,Manual,Semi-automatic,Unknown
fuel_group,,,,
Diesel,60.5,38.9,0.1,0.5
Electric,96.5,2.1,0.0,1.4
Hybrid,93.6,6.1,0.0,0.3
Other,39.3,59.9,0.0,0.7
Petrol,42.4,57.0,0.2,0.4


Electric cars are almost all automatic (the rest of the market is about half manual). Transmission is a consequence of the fuel type, not an
independent cause of depreciation, so we do **not** control for it (controlling for it would remove part of the effect we want to measure).

---
# 6. Insights

| # | Insight | What we do about it in notebook 03 |
|---|---|---|
| 1 | Prices are right-skewed | use the **log of the price** and **medians** |
| 2 | Electric cars are young (95% up to 6 years) | compare only the **age range 0-6 years**, where the groups overlap |
| 3 | Age and mileage are strongly correlated (0.83) | put **both** in the model |
| 4 | Electric cars start at a higher price | measure depreciation in **% per year**, not in euros |
| 5 | Electric cars are driven less (km per year) | control for mileage, with a **separate slope for each fuel** |
| 6 | Brand mix differs (VW, Renault, Hyundai, BMW...) | add **brand effects**, and repeat the test **inside the same model** (e.g. Fiat 500 electric vs petrol) |
| 7 | The raw median curves already show a steeper line for electric | test it properly, with confidence intervals |

Next: `03_hypothesis.ipynb`.